In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%cd /content/drive/MyDrive/nsmc-sentiment-analysis/bert/

import torch
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from torch.optim import AdamW
import json
from tqdm.auto import tqdm

from dataloader import get_loader
from config import BASE_DIR

train_loader, valid_loader, _ = get_loader()

/content/drive/MyDrive/nsmc-sentiment-analysis/bert


In [8]:
# 모델 생성
model = BertForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=2
)


# 계산 장치 지정
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
print(f"Using device: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda


In [9]:
# 체크포인트 저장 위치 지정
model_save_path = BASE_DIR / "bert" / "checkpoints" / f"best_bert.pt"

model_save_path.parent.mkdir(parents=True, exist_ok=True)

In [10]:
## 학습 및 검즘 ##

optimizer = AdamW(model.parameters(), lr=2e-5)

# 학습 과정 저장용 변수
train_losses = []
valid_losses = []

train_accs = []
valid_accs = []

# `best_loss_valid` 초기화
best_loss_valid = float('inf') # 가장 작은 valid loss를 추적하기 위해 무한대 값으로 초기화

for epoch in range(1, 6):

  # 학습 진행
  model.train()

  train_total_loss = 0
  train_correct = 0

  for batch in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    train_total_loss += loss.item()

    # 정확도 측정
    pred = outputs.logits.argmax(dim=1)
    train_correct += (pred == labels).sum().item()

  # 현재 에포크의 손실, 정확도 저장
  train_losses.append(train_total_loss / len(train_loader))
  train_accs.append(train_correct / len(train_loader.dataset))


  # 검증
  model.eval()

  valid_total_loss = 0
  valid_correct = 0

  with torch.no_grad():
    for batch in tqdm(valid_loader, desc=f"Epoch {epoch} Valid"):
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
      )

      loss = outputs.loss

      valid_total_loss += loss.item()

      # valid 정확도 측정
      pred = outputs.logits.argmax(dim=1)
      valid_correct += (pred == labels).sum().item()

  # 현재 에포크의 손실, 정확도 저장
  valid_losses.append(valid_total_loss / len(valid_loader))
  valid_accs.append(valid_correct / len(valid_loader.dataset))


  # 학습 과정 출력
  print(
      f"Epoch {epoch}/{5} | "
      f"Train Loss: {train_losses[-1]:.4f} | "
      f"Valid Loss: {valid_losses[-1]:.4f}"
  )


  # 체크포인트 저장
  if valid_losses[-1] < best_loss_valid:
      best_loss_valid = valid_losses[-1]

      torch.save(
          model.state_dict(),
          model_save_path
      )

Epoch 1 Train:   0%|          | 0/4219 [00:00<?, ?it/s]

Epoch 1 Valid:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 0.2768 | Valid Loss: 0.2616


Epoch 2 Train:   0%|          | 0/4219 [00:00<?, ?it/s]

Epoch 2 Valid:   0%|          | 0/469 [00:00<?, ?it/s]

...

Epoch 4 Valid:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 0.0800 | Valid Loss: 0.3196


Epoch 5 Train:   0%|          | 0/4219 [00:00<?, ?it/s]

Epoch 5 Valid:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 0.0599 | Valid Loss: 0.3855


In [11]:
# 학습 과정 저장
history_save_path = BASE_DIR / "bert"/ "results" / "history.json"

history_save_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

history = {
    "train_losses": train_losses,
    "valid_losses": valid_losses,
    "train_accs": train_accs,
    "valid_accs": valid_accs,
}

best_valid_loss = min(history["valid_losses"])
best_epoch = history["valid_losses"].index(best_valid_loss) + 1

history["best_valid_loss"] = best_valid_loss
history["best_epoch"] = best_epoch

with open(history_save_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=4)